# Generate and Evaluate Synthetic Data

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Synthetic Data

In [22]:
import pandas as pd
from torch.utils.data import DataLoader
from pathlib import Path
import evaluate
from openai import OpenAI
from datasets import Dataset, DatasetDict
from transformers import AdamW
from tqdm.auto import tqdm
from transformers import get_scheduler
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tools.utils import generate_response
from tools.prompt_templates import generate_negative_prompts, generate_positive_prompts
import pandas as pd
import re

In [25]:
clause_type = 'arbitration' #'arbitration' | 'opt-out' | 'class_waiver' 

In [26]:

processed_data_dir = Path('processed_data/multigenre')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')

In [29]:
clause_type = 'arbitration' #'arbitration' | 'opt-out' | 'class waiver' 
annotator = '_TS'
annotations_df = pd.read_excel(f'annotations/manual/{clause_type}_annotations_gpt4{annotator}.xlsx',
                               sheet_name=f'{clause_type}_annotations_gpt4', index_col=0)

annotations_df = annotations_df[~annotations_df.labels.isnull()]
annotations_df.labels = annotations_df.labels.astype(int)
annotations_df.replace({'labels': {2: 1}}, inplace=True)
annotations_df.labels.value_counts(normalize=True)

labels
1    0.833333
0    0.166667
Name: proportion, dtype: float64

In [30]:
annotations_df.drop_duplicates(subset='text', inplace=True)
annotations_df.shape

(96, 3)

In [31]:
def replace_mask_with_company(text):
    return re.sub(r'(\[mask\]\s)+', r'[company] ', text)

annotations_df['text'] = annotations_df.text.apply(replace_mask_with_company)

In [55]:

train,test = train_test_split(annotations_df, test_size=0.80, random_state=42)
train.labels.value_counts()

labels
1    16
0     3
Name: count, dtype: int64

## Generate Synthetic Data

In [ ]:
definitions = {
    'class waiver' : """A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that any dispute filed against each other must be on an individual basis and not as a class or collective action.""",
    'opt-out' : """a clause that permits signatories to a contract to opt out of particular provisions, or to terminate the contract early""",
    'arbitration': """In contract law, an arbitration clause is a clause in a contract that requires the parties to resolve their disputes through an arbitration process. Although such a clause may or may not specify that arbitration occur within a specific jurisdiction, it always binds the parties to a type of resolution outside the courts, and is therefore considered a kind of forum selection clause."""
}

num_examples = 100

In [19]:

positive_prompts = generate_positive_prompts(clause_type, definitions, train, num_examples)
negative_prompts = generate_negative_prompts(clause_type, definitions, train, num_examples)


In [ ]:
client = OpenAI()

df_postive =  pd.DataFrame(
                    [(p,generate_response(p,model='gpt-4o',max_tokens=100, temperature=0.5)) for p in tqdm(positive_prompts)],
                    columns=['prompt','text']
                    )
df_postive['labels'] = 1


0it [00:00, ?it/s]

In [ ]:
df_negative = pd.DataFrame(
                    [(p,generate_response(p,model='gpt-4o',max_tokens=100, temperature=0.5)) for p in tqdm(prompts_negative)],
                    columns=['prompt','text']
                    )
df_negative['labels'] = 0

In [ ]:
pd.concat([df_postive, df_negative], axis=0, ignore_index=True).to_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv')

# Train and evaluate classifier

In [46]:
df_sample = metadata.sample(n=200, random_state=0).reset_index(drop=True)
sents = df_sample.sentence.to_list()
sents = [s for s in sents if s not in train.text.to_list()]
df_sample = pd.DataFrame(sents, columns=['text'])
df_sample['labels'] = 0
df_sample.shape


(200, 2)

In [47]:
train_data = pd.concat([train[['text','labels']], df_sample[['text','labels']] ], axis=0, ignore_index=True).reset_index(drop=True)

## Add synthetic data

In [48]:
syntethic_data = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv', index_col=0)
syntethic_data

,prompt,text,labels
0,\n \nYou are a helpful AI that create...,"by entering into this agreement, you and [comp...",1
1,\n \nYou are a helpful AI that create...,"by entering into this contract, you and [compa...",1
2,\n \nYou are a helpful AI that create...,any disputes or claims related to this agreeme...,1
3,\n \nYou are a helpful AI that create...,any controversy or claim arising out of or rel...,1
4,\n \nYou are a helpful AI that create...,any dispute or claim arising from or connected...,1
...,...,...,...
195,You are a helpful AI that creates synthetic d...,"in the event that any disagreement arises, the...",0
196,You are a helpful AI that creates synthetic d...,any disagreements that are not covered by [com...,0
197,You are a helpful AI that creates synthetic d...,participation in any [company] initiative requ...,0
198,You are a helpful AI that creates synthetic d...,the interpretation and enforcement of this agr...,0


In [49]:
 
train_data = pd.concat([train_data, syntethic_data[['text','labels']]], axis=0, ignore_index=True).reset_index(drop=True)


In [50]:
train_data.labels.value_counts()

labels
0    304
1    120
Name: count, dtype: int64

## Train the model

In [53]:
import torch

def train_model(clause_type,train,test,checkpoint= "distilbert-base-uncased", num_epochs = 10):
    def tokenize_function(example):
        return tokenizer(example["text"], truncation=True)
    
    device = (
        "cuda"
        if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available() else "cpu"
        )

    torch.manual_seed(1984)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    # Combine into a DatasetDict
    dataset = DatasetDict({
        'train':  Dataset.from_pandas(train),
        'test': Dataset.from_pandas(test[['text','labels']].reset_index(drop=True)),
    })

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    tokenized_datasets = tokenized_datasets.remove_columns(["text"])
    tokenized_datasets.set_format("torch")

    train_dataloader = DataLoader(
        tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
            )
    eval_dataloader = DataLoader(
        tokenized_datasets["test"], batch_size=8, collate_fn=data_collator
        )
    
    optimizer = AdamW(model.parameters(), lr=1e-5,eps=1e-6, weight_decay=0.2) #eps=1e-6, weight_decay=0.2

    
    num_training_steps = num_epochs * len(train_dataloader)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=2,
        num_training_steps=num_training_steps,
    )

    progress_bar = tqdm(range(num_training_steps))
    metric = evaluate.load("glue", "mrpc")

    model.train()
    best_score = .0
    for epoch in range(num_epochs):
        for batch in train_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)


        model.eval()
        for batch in eval_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch["labels"])

        scores = metric.compute()
        print(f"Epoch {epoch}:", scores)

        if scores["f1"] > best_score:
            print("Saving model")
            best_score = scores["f1"] 
            model.save_pretrained(f"./models/{clause_type}_model")
            tokenizer.save_pretrained(f"./models/{clause_type}_model")
   


In [54]:
train_model(clause_type,train_data,test)

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/424 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/530 [00:00<?, ?it/s]

Epoch 0: {'accuracy': 0.7222222222222222, 'f1': 0.8333333333333334}
Saving model
Epoch 1: {'accuracy': 0.8472222222222222, 'f1': 0.916030534351145}
Saving model
Epoch 2: {'accuracy': 0.7916666666666666, 'f1': 0.88}
Epoch 3: {'accuracy': 0.8055555555555556, 'f1': 0.8870967741935484}
Epoch 4: {'accuracy': 0.8194444444444444, 'f1': 0.8943089430894309}
Epoch 5: {'accuracy': 0.8194444444444444, 'f1': 0.8943089430894309}
Epoch 6: {'accuracy': 0.8333333333333334, 'f1': 0.9032258064516129}
Epoch 7: {'accuracy': 0.8333333333333334, 'f1': 0.9032258064516129}
Epoch 8: {'accuracy': 0.8194444444444444, 'f1': 0.8943089430894309}
Epoch 9: {'accuracy': 0.8333333333333334, 'f1': 0.9032258064516129}


# Apply Classifier

In [ ]:
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
processed_data_dir = Path('processed_data/multigenre')
file_name = processed_data_dir / 'metadata.tsv'
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

149261


In [ ]:
clause_type = 'anti-scraping'
model = AutoModelForSequenceClassification.from_pretrained(f'models/{clause_type}_model')
tokenizer = AutoTokenizer.from_pretrained(f'models/{clause_type}_model')

In [ ]:
metadata.head()

,file_name,platform,genre,tag,sentence,logits_arbitration,prob_1_arbitration,arbitration
0,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,privacy policy effective date : [mask] [mask] ...,"[[tensor(0.9787), tensor(0.0213)]]",0.021337,0.0
1,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data controller is [mask] [mask] and can b...,"[[tensor(0.9763), tensor(0.0237)]]",0.023692,0.0
2,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data protection officer is : [mask] [mask]...,"[[tensor(0.9772), tensor(0.0228)]]",0.022761,0.0
3,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,[mask] ec4 m 7ef sciplaydpo@scientificgames.co...,"[[tensor(0.9687), tensor(0.0313)]]",0.031294,0.0
4,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,you may also use the link found at the end of ...,"[[tensor(0.9769), tensor(0.0231)]]",0.023066,0.0


In [ ]:
metadata.iloc[137375:137380]

,file_name,platform,genre,tag,sentence,logits_arbitration,prob_1_arbitration,arbitration
137375,tumblr_tos_social,tumblr,social,ToS,open source disclosures \n you can find disclo...,"[[tensor(0.9766), tensor(0.0234)]]",0.023355,0.0
137376,tumblr_tos_social,tumblr,social,ToS,,"[[tensor(0.7748), tensor(0.2252)]]",0.225161,0.0
137377,huggingface_tou_ai,huggingface,ai,ToS,thanks for using [mask] [mask] and being part ...,"[[tensor(0.9522), tensor(0.0478)]]",0.047831,0.0
137378,huggingface_tou_ai,huggingface,ai,ToS,we drafted the following [mask] [mask] [mask] ...,"[[tensor(0.9751), tensor(0.0249)]]",0.024918,0.0
137379,huggingface_tou_ai,huggingface,ai,ToS,we are very much open to feedback - contact us...,"[[tensor(0.9687), tensor(0.0313)]]",0.031342,0.0


In [ ]:
tqdm.pandas()
metadata[f'logits_{clause_type}'] = metadata.progress_apply(lambda x: 
                    softmax(model(**tokenizer(x.sentence, return_tensors='pt', truncation=True)).logits.detach(), dim=1), 
                          axis=1)

  0%|          | 0/149261 [00:00<?, ?it/s]

100%|██████████| 149261/149261 [1:12:32<00:00, 34.29it/s] 


In [ ]:
metadata[f'prob_1_{clause_type}'] = metadata[f'logits_{clause_type}'].apply(lambda x: x[0][1].item())
metadata[clause_type] = .0
metadata.loc[metadata[f'prob_1_{clause_type}'] > .5, clause_type] = 1

In [ ]:
metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.to_csv(processed_data_dir /f'{file_name.stem}_annotated.tsv', sep='\t', index=False)

In [ ]:
metadata.head()

,file_name,platform,genre,tag,sentence,prob_1_arbitration,arbitration,prob_1_anti-scraping,anti-scraping
0,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,privacy policy effective date : [mask] [mask] ...,0.021337,0.0,0.014945,0.0
1,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data controller is [mask] [mask] and can b...,0.023692,0.0,0.021128,0.0
2,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data protection officer is : [mask] [mask]...,0.022761,0.0,0.016152,0.0
3,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,[mask] ec4 m 7ef sciplaydpo@scientificgames.co...,0.031294,0.0,0.028223,0.0
4,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,you may also use the link found at the end of ...,0.023066,0.0,0.019758,0.0


In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence'])
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.text)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:


    df_deduplicated.loc[df_deduplicated.prob_1.between(*interval),'category']  = label



In [ ]:
pd.concat([df_deduplicated[df_deduplicated.category == label].sample(10)
    for _ , label in int_labels], axis=0)[['sentence','category']].to_csv(f'annotations/inference/{clause_type}_automatic_annotations_by_category.csv')


In [ ]:
metadata[metadata.prob_1 > .5].to_csv(f'annotations/inference/{clause_type}_inference.csv')